# Michigan Surtax Analysis: "Invest in Our Kids" Ballot Initiative

This notebook analyzes the proposed Michigan surtax from the "Invest in Our Kids" ballot initiative.

## Reform Details
- **Single filers**: 5% surtax on taxable income above $500,000
- **Joint filers**: 5% surtax on taxable income above $1,000,000
- **Effective date**: 2027

[Source: Invest in Our Kids ballot initiative](https://www.michigan.gov/sos/-/media/Project/Websites/sos/BSC-Announcements/Invest-in-MI-Kids-Petition.pdf)

In [1]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
from policyengine_core.charts import format_fig
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

## Setup Baseline and Reform Simulations

In [2]:
# Use Michigan state dataset
dataset = "hf://policyengine/policyengine-us-data/states/MI.h5"

# Baseline simulation
baseline = Microsimulation(dataset=dataset)

# Reform: Enable the Michigan surtax
reform = Reform.from_dict(
    {
        "gov.contrib.states.mi.surtax.in_effect": {
            "2027-01-01.2100-12-31": True
        }
    },
    country_id="us",
)

reformed = Microsimulation(reform=reform, dataset=dataset)

YEAR = 2027

## Revenue Impact

In [3]:
# Calculate the surtax revenue directly
mi_surtax = reformed.calc("mi_surtax", period=YEAR)
total_revenue = mi_surtax.sum()

print(f"Estimated annual revenue from Michigan surtax: ${total_revenue / 1e9:.2f} billion")
print(f"Number of tax units paying surtax: {(mi_surtax > 0).sum():,.0f}")
print(f"Average surtax among payers: ${mi_surtax[mi_surtax > 0].mean():,.0f}")

Estimated annual revenue from Michigan surtax: $5.69 billion
Number of tax units paying surtax: 52,297
Average surtax among payers: $108,888


## Who Pays the Surtax?

In [4]:
# Calculate share of population affected
total_tax_units = len(mi_surtax)
affected_tax_units = (mi_surtax > 0).sum()
share_affected = affected_tax_units / total_tax_units

print(f"Share of Michigan tax units affected: {share_affected:.2%}")
print(f"Total tax units in Michigan: {total_tax_units:,.0f}")
print(f"Tax units paying surtax: {affected_tax_units:,.0f}")

Share of Michigan tax units affected: 66.17%
Total tax units in Michigan: 79,036
Tax units paying surtax: 52,297


## Single vs Joint Filer Analysis

In [5]:
# Analyze by filing status
is_joint = baseline.calc("tax_unit_is_joint", period=YEAR)
mi_taxable_income = baseline.calc("mi_taxable_income", period=YEAR)

# Joint filers (threshold: $1M)
joint_mask = is_joint
joint_payers = mi_surtax[joint_mask] > 0
joint_affected_share = joint_payers.mean()
joint_revenue = mi_surtax[joint_mask].sum()
joint_count = joint_payers.sum()

# Single filers (threshold: $500k)
single_mask = ~is_joint
single_payers = mi_surtax[single_mask] > 0
single_affected_share = single_payers.mean()
single_revenue = mi_surtax[single_mask].sum()
single_count = single_payers.sum()

print("Joint Filers (threshold: $1,000,000):")
print(f"  Tax units affected: {joint_count:,.0f}")
print(f"  Share of joint filers affected: {joint_affected_share:.2%}")
print(f"  Revenue raised: ${joint_revenue / 1e9:.2f} billion ({joint_revenue/total_revenue*100:.1f}% of total)")
if joint_count > 0:
    print(f"  Average surtax: ${mi_surtax[joint_mask][mi_surtax[joint_mask] > 0].mean():,.0f}")
print()
print("Single Filers (threshold: $500,000):")
print(f"  Tax units affected: {single_count:,.0f}")
print(f"  Share of single filers affected: {single_affected_share:.2%}")
print(f"  Revenue raised: ${single_revenue / 1e9:.2f} billion ({single_revenue/total_revenue*100:.1f}% of total)")
if single_count > 0:
    print(f"  Average surtax: ${mi_surtax[single_mask][mi_surtax[single_mask] > 0].mean():,.0f}")

Joint Filers (threshold: $1,000,000):
  Tax units affected: 49,264
  Share of joint filers affected: 2.31%
  Revenue raised: $5.42 billion (95.2% of total)
  Average surtax: $110,064

Single Filers (threshold: $500,000):
  Tax units affected: 3,034
  Share of single filers affected: 0.10%
  Revenue raised: $0.27 billion (4.8% of total)
  Average surtax: $89,789


In [6]:
# Visualize revenue by filing status
filing_df = pd.DataFrame({
    "Filing Status": ["Joint Filers\n(>$1M threshold)", "Single Filers\n(>$500K threshold)"],
    "Revenue ($B)": [joint_revenue / 1e9, single_revenue / 1e9],
    "Tax Units Affected": [int(joint_count), int(single_count)]
})

fig = px.bar(
    filing_df,
    x="Filing Status",
    y="Revenue ($B)",
    title="Michigan Surtax Revenue by Filing Status (2027)",
    text="Revenue ($B)",
)
fig.update_traces(texttemplate="$%{text:.2f}B", textposition="outside")
fig.update_layout(yaxis=dict(tickformat="$,.2f"))
fig = format_fig(fig)
fig.show()

## Impact by Income Level

In [7]:
# Create income brackets for analysis
income_brackets = [
    (0, 100_000, "Under $100K"),
    (100_000, 250_000, "$100K-$250K"),
    (250_000, 500_000, "$250K-$500K"),
    (500_000, 1_000_000, "$500K-$1M"),
    (1_000_000, 2_000_000, "$1M-$2M"),
    (2_000_000, float('inf'), "Over $2M"),
]

bracket_results = []
for low, high, label in income_brackets:
    mask = (mi_taxable_income >= low) & (mi_taxable_income < high)
    count = mask.sum()
    surtax_in_bracket = mi_surtax[mask]
    payers = (surtax_in_bracket > 0).sum()
    revenue = surtax_in_bracket.sum()
    avg_surtax = surtax_in_bracket[surtax_in_bracket > 0].mean() if payers > 0 else 0
    
    bracket_results.append({
        "Income Bracket": label,
        "Tax Units": int(count),
        "Surtax Payers": int(payers),
        "Revenue ($M)": revenue / 1e6,
        "Avg Surtax ($)": avg_surtax
    })

bracket_df = pd.DataFrame(bracket_results)
bracket_df

,Income Bracket,Tax Units,Surtax Payers,Revenue ($M),Avg Surtax ($)
0,Under $100K,4739483,0,0.000000,0.000000
1,$100K-$250K,416927,0,0.000000,0.000000
2,$250K-$500K,105572,0,0.000000,0.000000
3,$500K-$1M,11008,411,5.252771,12751.214868
4,$1M-$2M,4436,4436,118.782595,26772.505501
5,Over $2M,47448,47448,5570.507272,117400.498843


In [8]:
# Visualize revenue by income bracket
fig = px.bar(
    bracket_df,
    x="Income Bracket",
    y="Revenue ($M)",
    title="Michigan Surtax Revenue by Income Bracket (2027)",
    text="Revenue ($M)",
)
fig.update_traces(texttemplate="$%{text:.0f}M", textposition="outside")
fig.update_layout(
    xaxis=dict(categoryorder="array", categoryarray=[b[2] for b in income_brackets]),
    yaxis=dict(tickformat="$,.0f"),
)
fig = format_fig(fig)
fig.show()

## Multi-Year Revenue Projection

In [9]:
# Project revenue over multiple years
years = range(2027, 2035)
revenue_results = []

for year in years:
    surtax = reformed.calc("mi_surtax", period=year)
    revenue = surtax.sum()
    payers = (surtax > 0).sum()
    revenue_results.append({
        "Year": year,
        "Revenue ($B)": revenue / 1e9,
        "Tax Units Paying": int(payers)
    })

revenue_df = pd.DataFrame(revenue_results)
revenue_df

MemoryError: Unable to allocate 557. KiB for an array with shape (142574,) and data type float32

In [ ]:
# Visualize multi-year revenue
fig = px.bar(
    revenue_df,
    x="Year",
    y="Revenue ($B)",
    title="Michigan Surtax: Projected Annual Revenue (2027-2034)",
    text="Revenue ($B)",
)
fig.update_traces(texttemplate="$%{text:.2f}B", textposition="outside")
fig.update_layout(
    xaxis=dict(tickmode="linear", tick0=2027, dtick=1),
    yaxis=dict(tickformat="$,.2f"),
)
fig = format_fig(fig)
fig.show()

In [ ]:
# Calculate 10-year total
ten_year_total = revenue_df["Revenue ($B)"].sum()
print(f"Total 8-year revenue (2027-2034): ${ten_year_total:.2f} billion")

## Surtax Distribution Analysis

In [ ]:
# Analyze distribution of surtax amounts among payers
payers_surtax = mi_surtax[mi_surtax > 0]

print("Distribution of Surtax Amounts Among Payers:")
print(f"  Minimum: ${payers_surtax.min():,.0f}")
print(f"  25th percentile: ${np.percentile(payers_surtax.values, 25):,.0f}")
print(f"  Median: ${payers_surtax.median():,.0f}")
print(f"  Mean: ${payers_surtax.mean():,.0f}")
print(f"  75th percentile: ${np.percentile(payers_surtax.values, 75):,.0f}")
print(f"  Maximum: ${payers_surtax.max():,.0f}")

In [ ]:
# Export results to CSV
revenue_df.to_csv("mi_surtax_revenue.csv", index=False)
bracket_df.to_csv("mi_surtax_income_brackets.csv", index=False)
filing_df.to_csv("mi_surtax_filing_status.csv", index=False)
print("Results exported to CSV files")

## Summary

The Michigan "Invest in Our Kids" surtax would:

- **Revenue**: Generate approximately $5.7 billion annually in 2027
- **Single filers**: 5% surtax on taxable income above $500,000
- **Joint filers**: 5% surtax on taxable income above $1,000,000
- **Affected population**: A small percentage of Michigan taxpayers (primarily high-income households)
- **Revenue source**: Majority comes from joint filers with income over $1 million

The revenue would be used for education funding as proposed in the ballot initiative.